# CADRE on Google Colab
**Paper:** *CADRE: A Context- and Domain-Aware Routing Engine for Dialogue State Tracking*

**Components:** FLAN-T5-large (SLM expert) + Gemini 2.5 Flash (LLM expert) + SenBERT KNN router + domain-aware reliability overlay

This notebook reproduces every configuration reported in the paper (Tables 1–3, Figures 2–4).

---
### ⚠️ Before you start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. You need a [Google AI Studio API key](https://aistudio.google.com/apikey) for the LLM (IC-DST) component
3. Runtime resets clear `/content` — use **Google Drive mounting** (Cell 2) to persist checkpoints


## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {total:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

## Cell 2 — Mount Google Drive (recommended to persist checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# All persistent data will live here — survives runtime resets.
# (Runs made before the rename used /content/drive/MyDrive/orchestrallm — point
#  DRIVE_DIR there to reuse existing checkpoints / pools.)
DRIVE_DIR = '/content/drive/MyDrive/cadre'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Working dir: {DRIVE_DIR}')

## Cell 3 — Install dependencies

In [ ]:
%%capture
!pip install transformers>=4.40.0 sentence-transformers>=2.6.0 \
             datasets>=2.18.0 accelerate>=0.27.0 google-genai>=1.0.0 \
             pyyaml tqdm
print('Done installing.')

## Cell 4 — Upload project files
Upload the Python files (`cadre.py`, `router.py`, `mrl_router.py`, `domain_router.py`, `ic_dst.py`, `prompt_dst.py`, `data_preprocessing.py`, `evaluate.py`, `calibrate_threshold.py`) + `config.yaml`, or pull from GitHub.


In [ ]:
# ── Option A: Upload files (Full or Partial) ──────────────────────────────────
# Modified to handle Colab's auto-renaming (e.g., 'file (1).py') and overwrite originals.

from google.colab import files
import os, shutil, re

PROJECT_DIR = '/content/cadre'
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Select files to upload (existing files will be overwritten):")
uploaded = files.upload()

for fname in uploaded:
    # 1. Strip the Colab suffix ' (1)' if it exists to find the intended filename
    # This regex looks for ' (number)' before the extension
    target_name = re.sub(r'\s\(\d+\)(?=\.\w+$)|\s\(\d+\)$', '', fname)

    dest_path = os.path.join(PROJECT_DIR, target_name)
    src_path = os.path.join('/content', fname)

    # 2. Move and overwrite
    if os.path.exists(src_path):
        shutil.move(src_path, dest_path)
        print(f'Updated: {dest_path}')

os.chdir(PROJECT_DIR)
print('\nCurrent files in project directory:')
!ls -lh {PROJECT_DIR}

In [ ]:
import os, re, shutil

PROJECT_DIR = '/content/cadre'
os.chdir(PROJECT_DIR)

# Pattern to match files with Colab version suffixes like ' (1)'
pattern = re.compile(r'.*\s\(\d+\)\.py$')

print('Cleaning up project directory...')
for fname in os.listdir(PROJECT_DIR):
    if pattern.match(fname):
        os.remove(os.path.join(PROJECT_DIR, fname))
        print(f'Removed redundant file: {fname}')

print('\nCleaned project directory:')
!ls -lh {PROJECT_DIR}

In [ ]:
# ── Option B: Pull from GitHub (if you've pushed the files) ──────────────────
# Uncomment and replace with your repo URL

# !git clone https://github.com/YOUR_USERNAME/cadre.git /content/cadre
# os.chdir('/content/cadre')

## Cell 5 — Set API key and configure paths

In [ ]:
import os
from getpass import getpass

# Set your Google AI Studio API key (only needed for IC-DST / LLM component)
os.environ['GEMINI_API_KEY'] = getpass('Enter your Google AI Studio API key: ')
print('API key set.')

In [ ]:
import yaml

DRIVE_DIR = '/content/drive/MyDrive/cadre'  # Persistent
PROJECT_DIR = '/content/cadre'               # Code lives here

# Update config.yaml to point paths to Google Drive (persistent across resets)
with open(f'{PROJECT_DIR}/config.yaml') as f:
    config = yaml.safe_load(f)

config['paths'] = {
    'data_dir':       f'{DRIVE_DIR}/data/multiwoz',
    'processed_dir':  f'{DRIVE_DIR}/data/processed',
    'model_dir':      f'{DRIVE_DIR}/models/prompt_dst',
    'expert_pool_dir':f'{DRIVE_DIR}/data/expert_pools',
    'results_dir':    f'{DRIVE_DIR}/results',
}

with open(f'{PROJECT_DIR}/config.yaml', 'w') as f:
    yaml.dump(config, f)

# Create all directories
for path in config['paths'].values():
    os.makedirs(path, exist_ok=True)

print('Paths configured:')
for k, v in config['paths'].items():
    print(f'  {k}: {v}')

## Cell 6 — Download MultiWOZ 2.4
This downloads the dataset directly from the official source (~50MB).

In [ ]:
import os, zipfile

DATA_DIR = config['paths']['data_dir']
os.makedirs(DATA_DIR, exist_ok=True)
data_file = f'{DATA_DIR}/data.json'

if os.path.exists(data_file):
    print(f'data.json already exists ({os.path.getsize(data_file)/1e6:.1f} MB) — skipping download.')
else:
    print('Downloading MultiWOZ 2.4...')
    # Download from the MultiWOZ 2.4 GitHub release
    !wget -q --show-progress -O /tmp/multiwoz24.zip \
        https://github.com/smartyfh/MultiWOZ2.4/raw/main/data/MULTIWOZ2.4.zip

    print('Extracting...')
    with zipfile.ZipFile('/tmp/multiwoz24.zip', 'r') as z:
        z.extractall('/tmp/multiwoz24/')

    # Find and move data.json
    import glob, shutil
    candidates = glob.glob('/tmp/multiwoz24/**/data.json', recursive=True)
    if candidates:
        shutil.copy(candidates[0], data_file)
        print(f'data.json saved → {data_file}')
    else:
        print('ERROR: data.json not found in zip. Files found:')
        !find /tmp/multiwoz24 -name '*.json' | head -20

    # Extract official split files (required for train/val/test splitting)
    for split_file in ['valListFile.json', 'testListFile.json']:
        matches = glob.glob(f'/tmp/multiwoz24/**/{split_file}', recursive=True)
        if matches:
            shutil.copy(matches[0], f'{DATA_DIR}/{split_file}')
            print(f'{split_file} saved → {DATA_DIR}/{split_file}')
        else:
            print(f'WARNING: {split_file} not found in zip')

import json
with open(data_file) as f:
    d = json.load(f)
print(f'Loaded {len(d)} dialogues from MultiWOZ 2.4')

## Cell 7 — Preprocess Data

In [ ]:
os.chdir(PROJECT_DIR)

processed_dir = config['paths']['processed_dir']
train_file = f'{processed_dir}/train.jsonl'

if os.path.exists(train_file):
    import subprocess
    n = int(subprocess.check_output(['wc', '-l', train_file]).split()[0])
    print(f'Preprocessed data already exists ({n} train turns) — skipping.')
else:
    print('Preprocessing MultiWOZ...')
    !python data_preprocessing.py \
        --data_dir  {config['paths']['data_dir']} \
        --out_dir   {config['paths']['processed_dir']} \
        --config    config.yaml

# Show split sizes
for split in ['train', 'holdout', 'val', 'test']:
    path = f"{config['paths']['processed_dir']}/{split}.jsonl"
    if os.path.exists(path):
        !echo -n "{split}: " && wc -l < {path} | xargs echo "turns"

## Cell 8 — Fine-tune Prompt-DST (FLAN-T5-large)
⏱️ **Estimated time:** ~2-4 hours on T4 GPU (5% few-shot data, 10 epochs)

💡 **Tip:** Colab Pro disconnects after ~12h. The best checkpoint is saved to Drive after each epoch — you can resume from there.

In [ ]:
model_dir = config['paths']['model_dir']
best_ckpt = f'{model_dir}/best'

if os.path.exists(f'{best_ckpt}/config.json'):
    print(f'Checkpoint already exists at {best_ckpt} — skipping training.')
    print('Delete it and rerun this cell to retrain.')
else:
    print('Starting FLAN-T5-large fine-tuning...')
    print('Checkpoints saved to Drive after each epoch.')
    !python prompt_dst.py train --config config.yaml

## Cell 9 — Evaluate Prompt-DST (SLM only)

In [ ]:
!python prompt_dst.py eval \
    --config config.yaml \
    --checkpoint {config['paths']['model_dir']}/best

## Cell 10 — Generate holdout predictions for both experts
This runs both SLM and LLM on the holdout set and saves predictions — needed to build expert pools.

In [ ]:
import sys, json
sys.path.insert(0, PROJECT_DIR)

from data_preprocessing import load_jsonl, string_to_state, format_input, state_to_string

pool_dir = config['paths']['expert_pool_dir']
os.makedirs(pool_dir, exist_ok=True)
processed_dir = config['paths']['processed_dir']

holdout_ex = load_jsonl(f'{processed_dir}/holdout.jsonl')
print(f'Holdout set: {len(holdout_ex)} turns')

In [ ]:
# ── SLM predictions on holdout ────────────────────────────────────────────────
slm_preds_path = f'{pool_dir}/slm_holdout_preds.json'

if os.path.exists(slm_preds_path):
    print('SLM holdout predictions already exist — skipping.')
else:
    import yaml, torch
    from prompt_dst import PromptDST
    from collections import defaultdict
    from tqdm.notebook import tqdm

    with open(f'{PROJECT_DIR}/config.yaml') as f:
        cfg = yaml.safe_load(f)

    slm = PromptDST(cfg)
    slm.load(f"{cfg['paths']['model_dir']}/best")

    # Group by dialogue to track accumulated DST correctly
    dialogues = defaultdict(list)
    for ex in holdout_ex:
        dialogues[ex['dialogue_id']].append(ex)
    for did in dialogues:
        dialogues[did].sort(key=lambda x: x['turn_idx'])

    slm_preds = {}
    for did, turns in tqdm(dialogues.items(), desc='SLM holdout'):
        acc_dst = {}
        for turn in turns:
            key = f"{turn['dialogue_id']}_{turn['turn_idx']}"
            inp = format_input(acc_dst, turn['agent_utt'], turn['user_utt'])
            pred_str = slm.predict_turn(inp)
            slm_preds[key] = pred_str
            acc_dst.update(string_to_state(pred_str))

    with open(slm_preds_path, 'w') as f:
        json.dump(slm_preds, f)
    print(f'SLM predictions saved ({len(slm_preds)} turns)')

In [ ]:
# ── LLM predictions on holdout (Gemini 2.5 Flash) ────
# Each turn sends fresh random 10 exemplars with full schema prompt.

llm_preds_path = f'{pool_dir}/llm_holdout_preds.json'

if os.path.exists(llm_preds_path):
    print('LLM holdout predictions already exist — skipping.')
else:
    import yaml
    from ic_dst import ICDST
    from collections import defaultdict
    from tqdm.notebook import tqdm

    with open(f'{PROJECT_DIR}/config.yaml') as f:
        cfg = yaml.safe_load(f)

    train_ex = load_jsonl(f"{cfg['paths']['processed_dir']}/train.jsonl")
    llm = ICDST(cfg)
    llm.load_exemplar_pool(train_ex)

    dialogues = defaultdict(list)
    for ex in holdout_ex:
        dialogues[ex['dialogue_id']].append(ex)
    for did in dialogues:
        dialogues[did].sort(key=lambda x: x['turn_idx'])

    llm_preds = {}
    total_turns = sum(len(t) for t in dialogues.values())
    with tqdm(total=total_turns, desc='LLM holdout calls') as pbar:
        for did, turns in dialogues.items():
            acc_dst = {}
            for turn in turns:
                key = f"{turn['dialogue_id']}_{turn['turn_idx']}"
                _, pred_str = llm.predict_turn(acc_dst, turn['agent_utt'], turn['user_utt'], did)
                llm_preds[key] = pred_str
                acc_dst.update(string_to_state(pred_str))
                pbar.update(1)

    with open(llm_preds_path, 'w') as f:
        json.dump(llm_preds, f)
    print(f'LLM predictions saved ({len(llm_preds)} turns)')

## Cell 11 — Build Expert Pools

In [ ]:
!python router.py build_pools --config config.yaml

## Cell 12 — (Optional, NOT in the paper) Contrastive fine-tuning of the SenBERT retriever
CADRE uses the **off-the-shelf** `all-mpnet-base-v2` encoder (paper Section 2.3, Appendix A). This step is inherited from OrchestraLLM and is **skipped by default**. If you run it, also set `router.use_fine_tuned_retriever: true` in `config.yaml`, otherwise `cadre.py` keeps using the off-the-shelf encoder.


In [ ]:
RUN_RETRIEVER_FINETUNE = False   # not a paper setting

retriever_path = f"{config['paths']['expert_pool_dir']}/retriever"
if not RUN_RETRIEVER_FINETUNE:
    print('Skipping retriever fine-tuning (paper uses the off-the-shelf encoder).')
elif os.path.exists(retriever_path):
    print('Retriever already fine-tuned — skipping.')
else:
    !python router.py train_retriever --config config.yaml


## Cell 12.5 — Calibrate the domain-aware override threshold τ (paper Section 2.5)
Sweeps τ ∈ {0.30, …, 0.80} on the holdout set by replaying the **cached** holdout predictions — no API calls. Selects the smallest τ whose override rate falls in the 5–15% band and writes it to `config.yaml` (paper: τ = 0.40).


In [ ]:
!python calibrate_threshold.py --config config.yaml --write --out {config['paths']['results_dir']}/threshold_sweep.json


## Cell 13 — Main results: the four configurations of Table 1
Each run evaluates the same slice of the validation set: whole dialogues in file order until **204 turns** (the paper's headline slice). Set `TURNS = 736` for the wider per-domain run of Section 3.3. Cost: the LLM-only run makes one Gemini call per turn.


In [ ]:
TURNS = 204          # paper headline slice; use 736 for the wide run (Section 3.3)
results_dir = config['paths']['results_dir']
os.makedirs(results_dir, exist_ok=True)

# Table 1, column 1 — SLM only
!python cadre.py eval --config config.yaml --max_turns {TURNS} --force_expert slm
# Table 1, column 2 — LLM only
!python cadre.py eval --config config.yaml --max_turns {TURNS} --force_expert llm
# Table 1, column 3 — base KNN router (no domain-aware overlay)
!python cadre.py eval --config config.yaml --max_turns {TURNS} --no_domain_aware
# Table 1, column 4 — CADRE (KNN router + domain-aware overlay)
!python cadre.py eval --config config.yaml --max_turns {TURNS}


In [ ]:
# Pretty-print Table 1 from the saved result files
import json, glob

rows = [('slm_only', 'SLM only'), ('llm_only', 'LLM only'),
        ('base_knn_router', 'Base KNN router'), ('knn_router+domain_aware', 'CADRE (full)')]
print(f"{'Configuration':<22}{'TLB JGA':>9}{'DST JGA':>9}{'Slot F1':>9}{'%SLM':>7}{'%LLM':>7}{'overr.':>8}{'sec':>8}")
for key, label in rows:
    path = f"{results_dir}/{key}_{TURNS}turns.json"
    if not os.path.exists(path):
        print(f"{label:<22}  (missing: {os.path.basename(path)})"); continue
    r = json.load(open(path))
    print(f"{label:<22}{r['tlb_jga']:>9.2f}{r['dst_jga']:>9.2f}{r['avg_slot_f1']:>9.2f}"
          f"{r['slm_assignment_ratio']:>7.1f}{r['llm_assignment_ratio']:>7.1f}"
          f"{r.get('domain_override_ratio', 0):>7.1f}%{r['eval_time_seconds']:>8.1f}")

print('\nPer-domain TLB JGA (Table 2):')
for key, label in rows:
    path = f"{results_dir}/{key}_{TURNS}turns.json"
    if os.path.exists(path):
        print(f"  {label:<20}", json.load(open(path)).get('per_domain_tlb_jga'))


## Cell 13.5 — Negative results (paper Section 4, Table 3)
**4.1 Margin-ranked learned router:** an MLP head on SenBERT predicting P(LLM), trained on the discriminative holdout subset (2 head-only warm-up epochs + 3 end-to-end, BCE with `pos_weight`), thresholded at 0.5.

**4.2 Semantically-retrieved in-context exemplars:** the LLM expert retrieves the K=10 most similar training exemplars with the same SenBERT encoder instead of sampling randomly.


In [ ]:
# 4.1 — train the MRL classifier head on the cached holdout predictions, then evaluate it
mrl_path = f"{config['paths']['expert_pool_dir']}/mrl_router"
if os.path.exists(f"{mrl_path}/head.pt"):
    print('MRL router already trained — skipping training.')
else:
    !python mrl_router.py train --config config.yaml

!python cadre.py eval --config config.yaml --max_turns {TURNS} --no_domain_aware --router mrl


In [ ]:
# 4.2 — base KNN router + semantic exemplars for the LLM expert
!python cadre.py eval --config config.yaml --max_turns {TURNS} --no_domain_aware --exemplar_selection semantic


In [ ]:
# Table 3
import json
rows = [('base_knn_router', 'Base KNN router'), ('knn_router+domain_aware', '+ Domain-aware'),
        ('knn_router+semantic_exemplars', '+ Semantic exemplars'), ('mrl_classifier', 'MRL classifier head')]
print(f"{'Configuration':<24}{'TLB':>8}{'DST':>8}{'F1':>8}{'%SLM':>7}")
for key, label in rows:
    path = f"{results_dir}/{key}_{TURNS}turns.json"
    if os.path.exists(path):
        r = json.load(open(path))
        print(f"{label:<24}{r['tlb_jga']:>8.2f}{r['dst_jga']:>8.2f}{r['avg_slot_f1']:>8.2f}{r['slm_assignment_ratio']:>7.1f}")
    else:
        print(f"{label:<24}  (missing)")


## Cell 13.6 — Figures 2 and 3
Per-domain gain over SLM-only, and the LLM-share vs DST-JGA trade-off. Configurations that have not been run yet fall back to the paper's reported numbers and are labelled *(paper)*.


In [ ]:
!python results/plot_paper_figures.py --results_dir {results_dir} --turns {TURNS}

from IPython.display import Image, display
display(Image(f"{results_dir}/figure2_per_domain_gain_{TURNS}turns.png"))
display(Image(f"{results_dir}/figure3_cost_quality_{TURNS}turns.png"))


## Cell 14 — Interactive Single-Turn Demo

In [ ]:
import sys, yaml
sys.path.insert(0, PROJECT_DIR)

from cadre import CADRE
from data_preprocessing import load_jsonl, state_to_string

with open(f'{PROJECT_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)

train_ex = load_jsonl(f"{cfg['paths']['processed_dir']}/train.jsonl")

pipeline = CADRE(cfg)
pipeline.load(exemplar_examples=train_ex)
print('Pipeline ready!')

In [ ]:
# ── Run a sample MultiWOZ dialogue (SNG01856) through CADRE ─────────────────
from data_preprocessing import string_to_state

# Simulating SNG01856.json turns
test_turns = [
    {
        'agent_utt': '',
        'user_utt':  'I am looking for a place to stay that has cheap price range, it should be a hotel.',
        'prev_dst':  {},
        'target':    'hotel-semi-type = hotel | hotel-semi-pricerange = cheap',
    },
    {
        'agent_utt': 'Okay, do you have a specific area you want to stay in?',
        'user_utt':  'No, I just need to make sure it is cheap. Oh, and I need parking.',
        'prev_dst':  {'hotel-semi-type': 'hotel', 'hotel-semi-pricerange': 'cheap'},
        'target':    'hotel-semi-parking = yes',
    },
    {
        'agent_utt': 'I found 1 cheap hotel for you that includes parking. Shall I book it?',
        'user_utt':  'Yes please. 6 people, 3 nights starting on Tuesday.',
        'prev_dst':  {'hotel-semi-type': 'hotel', 'hotel-semi-pricerange': 'cheap', 'hotel-semi-parking': 'yes'},
        'target':    'hotel-book-people = 6 | hotel-book-stay = 3 | hotel-book-day = tuesday',
    },
]

print('Running CADRE on example dialogue from paper (SNG01856)...\n')
accumulated_dst = {}

for i, turn in enumerate(test_turns):
    expert, pred_tlb, pred_str = pipeline.predict_turn(
        prev_dst=accumulated_dst,
        agent_utt=turn['agent_utt'],
        user_utt=turn['user_utt'],
    )
    accumulated_dst.update(pred_tlb)

    print(f'── Turn {i+1} ──')
    print(f'  User    : {turn["user_utt"]}')
    print(f'  Expert  : {expert.upper()}')
    print(f'  Predicted TLB : {pred_str or "[none]"}')
    print(f'  Gold TLB      : {turn["target"]}')
    print(f'  Full DST      : {state_to_string(accumulated_dst)}')
    print()

## Cell 15 — Download results from Colab
If not using Drive, download results to your local machine.

In [ ]:
from google.colab import files
import shutil, os

results_dir = config['paths']['results_dir']
if os.path.isdir(results_dir) and os.listdir(results_dir):
    shutil.make_archive('/content/cadre_results', 'zip', results_dir)
    files.download('/content/cadre_results.zip')
else:
    print('No results found. Run Cell 13 first.')


---
## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Reduce `batch_size` in config.yaml (try 4) or `gradient_accumulation_steps: 8` |
| `Runtime disconnected` | Checkpoints are on Drive — just reconnect and re-run from Cell 8 (it will skip if checkpoint exists) |
| `google.genai` authentication error | Re-run Cell 5 to re-enter your API key (it doesn't persist across sessions) |
| `data.json not found` | MultiWOZ zip structure may have changed — manually upload `data.json` to Drive path |
| Slow training | Normal for T4 — consider Colab Pro for A100 (~4x faster) |